# 第11章：時系列データ（`DatetimeIndex / resample / rolling / shift`）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
DATA = Path.cwd() / "assets"
sales = pd.read_csv(DATA / "sales_sample.csv")
# 不正な日付をNaTにしてから使う
sales['date'] = pd.to_datetime(sales['date'], errors='coerce')

## 11.1 DatetimeIndex と set_index

In [ ]:
ts = (sales.dropna(subset=['date'])
            .set_index('date')
            .sort_index())
ts.head()

## 11.2 resample（月・週・日）

In [ ]:
monthly = ts['amount'].resample('M').sum()
weekly = ts['amount'].resample('W').sum()
daily = ts['amount'].resample('D').sum()
monthly.head(), weekly.head(), daily.head()

In [ ]:
# プロット（保存）
fig = plt.figure()
monthly.plot()
plt.title('Monthly Sales')
plt.xlabel('Month'); plt.ylabel('Amount')
out = Path('assets') / 'ts_monthly.png'
fig.savefig(out, bbox_inches='tight')
out

## 11.3 rolling（移動平均）

In [ ]:
ma7 = daily.rolling(7).mean()
ma30 = daily.rolling(30).mean()
fig2 = plt.figure()
daily.plot(label='daily')
ma7.plot(label='ma7')
ma30.plot(label='ma30')
plt.title('Daily Sales with Moving Averages')
plt.xlabel('Date'); plt.ylabel('Amount')
out2 = Path('assets') / 'ts_moving_avg.png'
fig2.savefig(out2, bbox_inches='tight')
out2

## 11.4 shift（前年比/前週比の計算）

In [ ]:
m = monthly.to_frame('amount')
m['amount_prev_year'] = m['amount'].shift(12)
m['yoy'] = (m['amount'] - m['amount_prev_year']) / m['amount_prev_year']
m.head(15)

## 小課題：店舗別の月次売上と前年同月比

In [ ]:
# ヒント：storeごとに月次へresample→shift(12)で前年比
store_m = (ts.groupby('store')['amount']
             .resample('M').sum()
             .unstack('store'))
yoy = store_m.pct_change(12)
store_m.head(3), yoy.head(15)